# Cilia manual classification
The purpose of this notebook is to classify mesenchymal and ductal cilia in our entire dataset. Here we use the Morphocilia segmentation and a manual classification on the Napari interface to create a dataset containing the properties of the classified cilia. Then, seaborn is used to check for differences between mesenchymal and ductal cilia, and the changes of both classes throughout stages.

In [ ]:
from aicsimageio import AICSImage
import matplotlib.pyplot as plt
from morphocilia.io import load_rescaled_channel
from morphocilia.segmenter import cilia_segmenter_cleaner
import pandas as pd
from pathlib import Path
import seaborn as sns
from skimage.io import imread
from skimage.measure import regionprops_table

In [ ]:
# Path of the different images
DATA_DIR = Path("../data")
filepath = (
    (DATA_DIR / "20231011_e18_cd13_opn_arl13.lif"),
    (DATA_DIR / "20231025_p1_cd13_opn_arl13b.lif"),
    (DATA_DIR / "20231108_p7_cd13_opn_arl13b.lif"),
    (DATA_DIR / "20231122_p15_cd13_opn_arl13b.lif"),
)

In [ ]:
# Path of the manually classified labels
CLASS_DIR = Path("../data/classification")
classpath = (
    (CLASS_DIR / "e18_pos1_hilar_0.tif"),
    (CLASS_DIR / "e18_pos2_hilar_1.tif"),
    (CLASS_DIR / "e18_pos3_hilar_2.tif"),
    (CLASS_DIR / "p1_pos1_zstack_1.tif"),
    (CLASS_DIR / "p1_pos2_zstack_5.tif"),
    (CLASS_DIR / "p1_pos3_zstack_7.tif"),
    (CLASS_DIR / "p1_pos4_zstack_8.tif"),
    (CLASS_DIR / "p1_pos5_zstack_9.tif"),
    (CLASS_DIR / "p7_pos1_zstack_1.tif"),
    (CLASS_DIR / "p7_pos2_zstack_2.tif"),
    (CLASS_DIR / "p7_pos3_zstack_3.tif"),
    (CLASS_DIR / "p15_pos2_zstack_1.tif"),
    (CLASS_DIR / "p15_pos3_zstack_2.tif"),
    (CLASS_DIR / "p15_pos4_zstack_3.tif"),
)

In [ ]:
# For loop to select the 3D scenes of our images and append them in a list
scenes = []
for file in filepath:
    if file.suffix != ".lif":
        continue

    file_handle = AICSImage(file)

    for scene in file_handle.scenes:
        print(scene)
        file_handle.set_scene(scene)
        if file_handle.dims["Z"][0] <= 1:
            continue

        print(file_handle.dims)
        scenes.append((file, scene))

In [ ]:
# Function to change the classes labels from numbers to class type names
def parse_classes(class_number):
    if class_number == 1:
        return "ductal"
    elif class_number == 2:
        return "mesenchymal"
    elif class_number == 0:
        return "NA"
    else:
        raise ValueError

In [ ]:
# Function to relate each image with its developmental stage
def parse_stages(image_name):
    if "e18" in str(image_name):
        return "e18"
    elif "p1" in str(image_name):
        return "p1"
    elif "p7" in str(image_name):
        return "p7"
    elif "p15" in str(image_name):
        return "p15"
    else:
        raise ValueError

In [ ]:
# For loop that takes the image, the scene, and its corresponding manual classification and makes a dataframe with preselected properties, image name, scene name, cilia type, and stage
df_all = pd.DataFrame()
for (image_path, scene), file in zip(scenes, classpath):
    # Scene is loaded, segmented, and labelled
    cilia_channel = load_rescaled_channel(
        image_path,
        scene,
        1,
    )

    labelled_prediction = cilia_segmenter_cleaner(cilia_channel)

    # A dataframe with selected properties, image name, and scene name is created
    this_df = regionprops_table(
        labelled_prediction,
        cilia_channel.compute(),
        properties=[
            "axis_major_length",
            "axis_minor_length",
            "extent",
            "intensity_max",
            "intensity_min",
            "solidity",
        ],
        spacing=(0.3, 0.144, 0.144),  # i.e. scale
    )
    this_df["image_name"] = image_path
    this_df["scene_name"] = scene
    this_df = pd.DataFrame(this_df)

    # The manual classification is loaded and two more columns are added to the dataframe: label, and cilia type
    manual_classification = imread(file)
    props_class = regionprops_table(
        labelled_prediction,
        manual_classification,
        properties=[
            "label",
            "intensity_max",
        ],  # Intensity max in this case means 1 or 2, which corresponds to classes in manual classification.
    )
    this_df[["label", "cilia_type"]] = pd.DataFrame(props_class).rename(
        columns={"intensity_max": "cilia_type"}
    )
    # The functions parse_classes and parse_stages are applied
    this_df["cilia_type"] = this_df["cilia_type"].apply(parse_classes)
    this_df["stage"] = this_df["image_name"].apply(parse_stages)
    # The dataframes created for each scene are concatenated in a single dataframe
    df_all = pd.concat([df_all, this_df], ignore_index=True)
df_all

In [ ]:
# A new dataframe is created by filtering the main dataframe for cilia that are classified (NA are ignored)
df_classified = df_all.query("cilia_type != 'NA'")
df_classified

In [ ]:
# A couple of cilia wrongly classified are eliminated
df_classified = df_classified.drop([92, 1337])
df_classified

In [ ]:
# The dataframe is saved as a .csv
df_classified.to_csv("manual_classification_cilia.csv")

In [ ]:
# Comparison of mesenchymal vs ductal cilia irrespective of stage or bile duct size
props_to_plot = [
    "axis_major_length",
    "axis_minor_length",
    "extent",
    "intensity_max",
    "intensity_min",
    "solidity",
]

fig, axs = plt.subplots(2, 3, figsize=(20, 10))
axs = axs.flatten()

for ax, prop in zip(axs, props_to_plot):
    sns.violinplot(data=df_classified, x="cilia_type", y=prop, ax=ax)

In [ ]:
# New dataframe containing only mesenchymal cilia
df_classified_mesenchymal = df_classified.query("cilia_type == 'mesenchymal'")
df_classified_mesenchymal

In [ ]:
# New dataframe containing only ductal cilia
df_classified_ductal = df_classified.query("cilia_type == 'ductal'")
df_classified_ductal

In [ ]:
# Changes in mesenchymal cilia properties throughout stages

props_to_plot = [
    "axis_major_length",
    "axis_minor_length",
    "extent",
    "intensity_max",
    "intensity_min",
    "solidity",
]

fig, axs = plt.subplots(2, 3, figsize=(20, 10))
axs = axs.flatten()

for ax, prop in zip(axs, props_to_plot):
    sns.violinplot(data=df_classified_mesenchymal, x="stage", y=prop, ax=ax)

In [ ]:
# Changes in ductal cilia properties throughout stages
props_to_plot = [
    "axis_major_length",
    "axis_minor_length",
    "extent",
    "intensity_max",
    "intensity_min",
    "solidity",
]

fig, axs = plt.subplots(2, 3, figsize=(20, 10))
axs = axs.flatten()

for ax, prop in zip(axs, props_to_plot):
    sns.violinplot(data=df_classified_ductal, x="stage", y=prop, ax=ax)

Extent and axis_minor_length seem to be the best properties to characterise both mesenchymal and ductal cilia.